Parameter ablation primary figures.

In [ ]:

import json
import os
import random
import importlib
from collections import defaultdict
import sys
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr
from tqdm import tqdm

import analysis_utils
from analysis_utils import tidy_game_code
import constants

main_dir = constants.PROCESSED_RES_DIR
main_save = constants.FINAL_FIGURES_DIR
fig_data_file_pth = constants.FIG_DATA_DIR


import matplotlib as mpl 
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
base_game_objs, idx2game, game2idx, game_stimuli = (
    analysis_utils.process_game_stimuli(constants.THINK_STIMULI_PTH)
)
human_df = pd.read_csv(constants.THINK_HUMAN_DATA)
all_game_types, game2game_type = analysis_utils.get_game_categories(human_df)


In [ ]:

model2name = dict(constants.PAPER_MODEL2NAME)
agent2color = dict(constants.PAPER_AGENT2COLOR)
game_type2fmt = constants.PAPER_GAME_TYPE2FMT
order_game_types = list(constants.PAPER_ORDER_GAME_TYPES)
model2color = {}


In [ ]:
'''    
Load param ablations
'''

param_fit_dir = "final_processed_res/param_fits/"

think_params_df = pd.read_csv(f"{param_fit_dir}/think_r2.csv")
watch_params_df = pd.read_csv(f"{param_fit_dir}/watch_tvd.csv")
with open(f"play_paramfit.json", "r") as f: 
    play_params_df = json.load(f)['individ']

type2param_df = {'think': {'df': think_params_df, 'metric': 'R²'}, 'play': {'df': play_params_df, 'metric': 'Aggregate Move Likelihood'}, 'watch': {'df': watch_params_df, 'metric': 'TVD'},}


viz_agents = list(play_params_df.keys())
fixed_alpha_res = play_params_df

In [ ]:
''' 
Load play data
'''


agg_scores_alpha= {}
individ_move_scores_alpha = {}
individ_move_scores_per_alpha_agent = {}


move_idx_scores_per_alpha_agent = {agent: {} for agent in viz_agents}
individ_move_scores_alpha_by_game = {agent: defaultdict(list) for agent in viz_agents}

individ_move_scores_per_alpha_per_game = {}


for agent_idx, agent in enumerate(viz_agents): 
    collapsed_scores = []
    alpha_vals = sorted(list(fixed_alpha_res[agent].keys()))
        
    individ_move_scores_alpha_by_game[agent] = defaultdict(list)
    
    individ_logprobs, game_metadata = fixed_alpha_res[agent][alpha_vals[0]][0] # [logprobs, games]
    print(len(individ_logprobs))
    n_matches = len(individ_logprobs) # take one to get shape for agg
    match_scores = np.zeros(n_matches)
    individ_move_scores = np.zeros_like(np.array(sum(individ_logprobs, [])))
    
    individ_move_scores_per_alpha= np.zeros([len(alpha_vals), len(np.array(sum(individ_logprobs, [])))])
    
    kept_alpha = set()
    
    
    
    for alpha_idx, alpha in enumerate(alpha_vals): 
        alpha_float = float(alpha)

        curr_move_idx = 0
        
        move_idx2game = {}
        
        individ_logprobs, game_metadata = fixed_alpha_res[agent][alpha][0]
        ran_games = set([game for game, _ in game_metadata])
        
        print(len(ran_games))
        
        game_list = []
        for game, arenas_in_game in game_metadata:
            if len(arenas_in_game) == 0: print("missing game: ", game) 
            game_list.extend([game for _ in range(len(arenas_in_game) * 2)]) # b/c per individ
        
        
        for match_idx in range(n_matches): 
            
            match_vals = individ_logprobs[match_idx]
            if np.any(['inf' in str(x) for x in match_vals]): print("inf " + match_vals)
            curr_game = game_list[match_idx] 
                
            match_scores[match_idx] += sum(match_vals)
            for move_idx, move_val in enumerate(match_vals): 
                move_idx2game[curr_move_idx] = curr_game 
                individ_move_scores[curr_move_idx] += move_val
                individ_move_scores_per_alpha[alpha_idx, curr_move_idx] = move_val
                curr_move_idx+= 1

                move_idx_scores_per_alpha_agent[agent].setdefault(alpha, {})
                move_idx_scores_per_alpha_agent[agent][alpha].setdefault(move_idx, [])
                move_idx_scores_per_alpha_agent[agent][alpha][move_idx].append(move_val)
                
            kept_alpha.add(alpha)
    match_scores /= len(kept_alpha)
    individ_move_scores /= len(kept_alpha)
    
    agg_scores_alpha[agent] = match_scores #collapsed_scores
    individ_move_scores_alpha[agent] = individ_move_scores
    
    
    individ_move_scores_per_alpha_per_game[agent] = {}
    for move_idx in range(curr_move_idx): 
        game = move_idx2game[move_idx]
        individ_move_scores_per_alpha_per_game[agent].setdefault(game, [])
        individ_move_scores_per_alpha_per_game[agent][game].append(individ_move_scores[move_idx])
    
    
    individ_move_scores_per_alpha_agent[agent] = individ_move_scores_per_alpha


match_counts = 0
for game, arenas in game_metadata: 
    match_counts += len(arenas) * 2
print(match_counts)




In [ ]:

plt.figure(figsize=(12, 6))

config2name = {"{'center_weight': 1.0, 'connect_weight': 1.0, 'block_weight': 1.0}":  'All Components',
 "{'center_weight': 0.0, 'connect_weight': 1.0, 'block_weight': 1.0}": 'Ablate Center Weight',
 "{'center_weight': 1.0, 'connect_weight': 0.0, 'block_weight': 1.0}": 'Ablate Connect Weight',
 "{'center_weight': 1.0, 'connect_weight': 1.0, 'block_weight': 0.0}": 'Ablate Block Weight',
 "{'center_weight': 0.0, 'connect_weight': 0.0, 'block_weight': 1.0}": 'Only Block Weight',
 "{'center_weight': 0.0, 'connect_weight': 1.0, 'block_weight': 0.0}": 'Only Connect Weight',
 "{'center_weight': 1.0, 'connect_weight': 0.0, 'block_weight': 0.0}": 'Only Center Weight'}


n_boot = 1000

sub_vizagents = [agent for agent in viz_agents if 'depth3' not in agent]

# Calculate bootstrapped statistics for each agent
agent_stats = {}
updated_df = {'Configuration': [], 'mean': [], 'ci_lower': [], 'ci_upper': []}

for agent in sub_vizagents:
    # Get all scores for this agent across all alpha values
    vals = individ_move_scores_alpha[agent]
    # Apply bootstrap to get statistics
    mean, lower, upper = analysis_utils.bootstrap_column_subsample(vals, n_boot=n_boot) 
    
    agent_stats[agent] = {
        'mean': mean[0],
        'lower': lower[0],
        'upper': upper[0],
    }
    
    updated_df['Configuration'].append(config2name[agent])
    updated_df['mean'].append(agent_stats[agent]['mean'])
    updated_df['ci_lower'].append(agent_stats[agent]['lower'])
    updated_df['ci_upper'].append(agent_stats[agent]['upper'])

# Prepare data for barplot
agents = [config2name[agent].replace(" ", "\n") for agent in sub_vizagents]
means = [agent_stats[agent]['mean'] for agent in sub_vizagents]
colors = ['grey' for agent in sub_vizagents]

plt.bar(agents, means, color=colors, alpha=0.7, edgecolor='black', linewidth=2,
        width=0.6)

# Add error bars using 95% CI from bootstrap
yerr = [
    [agent_stats[agent]['mean'] - agent_stats[agent]['lower'] for agent in sub_vizagents],  # lower errors
    [agent_stats[agent]['upper'] - agent_stats[agent]['mean'] for agent in sub_vizagents]   # upper errors
]
plt.errorbar(agents, means, yerr=yerr, fmt='none', ecolor='black', capsize=10, linewidth=3)#1.5)


plt.xlabel("", fontsize=18)
plt.ylabel("Move Log Likelihood", fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=20)
plt.gca().invert_yaxis()  
plt.ylim([-2.0, -3.3])
plt.tight_layout()

# Save the figure
# plt.savefig('final_figures/play/aggregate_logprobs_bar_bootstrap.png', dpi=400)
updated_df = pd.DataFrame(updated_df)
type2param_df['play']['df'] = updated_df


In [ ]:


goal_offense = 'Goal Offense'
goal_defense = 'Goal Defense'
center_txt = 'Center Bias'
component_map = {
    'All Components': [goal_offense, goal_defense, center_txt],
    'Ablate Center Weight': [goal_offense, goal_defense,],
    'Ablate Connect Weight': [goal_defense, center_txt],
    'Ablate Block Weight': [goal_offense,  center_txt],
    'Only Connect Weight': [goal_offense],
    'Only Block Weight': [goal_defense],
        'Only Center Weight': [center_txt],
}
components = [goal_offense, goal_defense, center_txt]
config_order = list(component_map.keys())

hatch_patterns = ['////', '\\\\\\\\', 'xxxx', '....', 'ooo', '***', '---']

exp2lims = {'play': [-2, -3.1], 'watch': [0.35, 0.525], 'think': [0, 0.85]}

for i, (exp_type, exp_data) in enumerate(type2param_df.items()):
    violin_df = exp_data['df']
    metric = exp_data['metric']
    
    fig, ax = plt.subplots(figsize=(8, 6))

    if exp_type == 'play': 
        summary = violin_df.set_index('Configuration').loc[config_order][['mean', 'ci_lower', 'ci_upper']]
        metric = 'Agg Move Likelihood'
    else: 
        if exp_type == 'watch':
            grouped = violin_df.groupby('Configuration')['TV']
            metric = 'Total Variation Distance'
            n_boot = 1000  

            for config in config_order:
                if config in grouped.groups:
                    group_data = grouped.get_group(config).dropna().values
                    means = []
                    for _ in range(n_boot):
                        sample = np.random.choice(group_data, size=len(group_data), replace=True)
                        means.append(np.mean(sample))
                    mean = np.mean(means)
                    ci_lower = np.percentile(means, 2.5)
                    ci_upper = np.percentile(means, 97.5)
                    summary.loc[config, 'mean'] = mean
                    summary.loc[config, 'ci_lower'] = ci_lower
                    summary.loc[config, 'ci_upper'] = ci_upper
                else:
                    summary.loc[config] = [np.nan, np.nan, np.nan]
            summary = summary.astype(float)

        else: 
            grouped = violin_df.groupby('Configuration')['R² Score']
            summary = pd.DataFrame(index=config_order, columns=['mean', 'ci_lower', 'ci_upper'])
            for config in config_order:
                group_data = grouped.get_group(config)
                mean = group_data.mean()
                ci_lower = np.percentile(group_data, 2.5)
                ci_upper = np.percentile(group_data, 97.5)
                summary.loc[config, 'mean'] = mean
                summary.loc[config, 'ci_lower'] = ci_lower
                summary.loc[config, 'ci_upper'] = ci_upper
            summary = summary.astype(float)
    
    x = np.arange(len(config_order))
    bar_width = 0.6
    yerr = np.array([
        summary['mean'] - summary['ci_lower'],
        summary['ci_upper'] - summary['mean']
    ])
    
    for j, config in enumerate(config_order):
        ax.bar(
            x[j],
            summary.loc[config, 'mean'],
            color='grey',
            width=bar_width,
            hatch=hatch_patterns[j],
            edgecolor='black',
            alpha=0.5,
            label=config if j == 0 else "" 
        )
    ax.errorbar(x, summary['mean'], yerr=yerr, fmt='none', ecolor='black', capsize=10, zorder=10)
    
    save_viz_data = pd.DataFrame({'means': summary['mean'], 'err_low': yerr[0], 'err_high': yerr[1], 'config': config_order})
    save_viz_data.to_csv(fig_data_file_pth + f'{exp_type}_params.csv')
    
    
    ax.set_ylabel(metric, fontsize=20)
    ax.set_xticks(x)
    ax.set_xticklabels(['' for _ in config_order], rotation=45, ha='right', fontsize=12)
    ax.tick_params(axis='y', labelsize=14)
    ax.set_xlabel('')
    
    legend_handles = [
        Patch(facecolor='grey', edgecolor='black', hatch=hatch_patterns[i], label=config_order[i])
        for i in range(len(config_order))
    ]

    if exp_type == 'play': 
        y_max = -3.2
        y_min = -0.5
    else: 
        y_max = float(max(summary['ci_upper']) * 1.1)
        y_min = 0.3
        
    y_min, y_max = exp2lims[exp_type]
        
    ax.set_ylim(y_min, y_max)
    
    # dot grid for component presence
    fig.subplots_adjust(bottom=0.28)
    ax_dots = fig.add_axes([0.125, 0.02, 0.775, 0.18])
    
    for k, comp in enumerate(components):
        for j, config in enumerate(config_order):
            is_present = comp in component_map[config]
            color = 'black' if is_present else 'lightgray'
            ax_dots.scatter(j, len(components)-1-k, color=color, s=80)
    
    ax_dots.set_xlim(-0.5, len(config_order) - 0.5)
    ax_dots.set_ylim(-0.5, len(components) - 0.5)
    ax_dots.set_xticks(x)
    ax_dots.set_xticklabels([], rotation=45, ha='right', fontsize=11)
    ax_dots.set_yticks(range(len(components)))
    ax_dots.set_yticklabels(components[::-1], fontsize=13)
    ax_dots.tick_params(left=False, bottom=False)
    for spine in ax_dots.spines.values():
        spine.set_visible(False)

    
    
    plt.savefig(f'final_figures/param_fits/ablation_bar_dotgrid_95ci_{exp_type}.svg', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
agg_metrics = pd.read_csv("final_processed_res/play/acc_df.csv")

# convert source name: 
source_name2orig = {'Novice': 'ours', 'Random':'random', 'Expert': 'expert', 'Novice\n(Depth-3)': 'ours-depth3'}
agg_metrics['source'] = [source_name2orig[src] for src in agg_metrics['source']]


In [ ]:
# Load saved_r2 produced by figures_think.ipynb (cell 8 there).
# These R^2 bootstrap distributions are needed by the next cell
# to fill in the Temp=0 and Depth-3 ablation configs.
with open(f"{main_dir}think/saved_r2.json") as f:
    saved_r2 = json.load(f)


In [ ]:


# setup components
goal_offense = 'Goal Offense?'
goal_defense = 'Goal Defense?'
probabilistic_label = 'Probabilistic?'
flatness_label = 'Flat?'

component_map = {
    'All Components': [goal_offense, goal_defense, probabilistic_label, flatness_label],
    'Depth-3': [goal_offense, goal_defense, probabilistic_label],
    'Temp=0': [goal_offense, goal_defense, flatness_label],
    'Ablate Block Weight': [goal_offense, probabilistic_label, flatness_label],
    'Ablate Connect Weight': [goal_defense, probabilistic_label, flatness_label],
    'Only Center Weight': [probabilistic_label, flatness_label],
}

components = [goal_offense, goal_defense, probabilistic_label, flatness_label]
config_order = list(component_map.keys())

# loop through experiment types
for i, (exp_type, exp_data) in enumerate(type2param_df.items()):
    if exp_type != 'think':
        continue

    violin_df = exp_data['df']
    metric = exp_data['metric']

    if metric == '-TVD':
        violin_df['R² Score'] = [-s for s in violin_df['R² Score']]

    fig, ax = plt.subplots(figsize=(7, 5)) 

    grouped = violin_df.groupby('Configuration')['R² Score']
    summary = pd.DataFrame(index=config_order, columns=['mean', 'ci_lower', 'ci_upper'])

    for config in config_order:
        if config == 'Temp=0':
            values = saved_r2['temp0-ours']
        elif config == 'Depth-3':
            values = saved_r2['depth3']
        else:
            values = grouped.get_group(config).values

        summary.loc[config, 'mean'] = np.mean(values)
        summary.loc[config, 'ci_lower'] = np.percentile(values, 2.5)
        summary.loc[config, 'ci_upper'] = np.percentile(values, 97.5)

    x = np.arange(len(config_order))
    bar_width = 0.6
    hatch_patterns = ['////', '\\\\\\\\', 'xxxx', '....', 'ooo', '----']


    for i, config in enumerate(config_order):
        ax.bar(
            x[i],
            summary.loc[config, 'mean'],
            color='grey',
            width=bar_width,
            hatch=hatch_patterns[i],
            edgecolor='black',
            alpha=0.5
        )

        ax.errorbar(
            x[i],
            summary.loc[config, 'mean'],
            yerr=[[summary.loc[config, 'mean'] - summary.loc[config, 'ci_lower']],
                  [summary.loc[config, 'ci_upper'] - summary.loc[config, 'mean']]],
            fmt='none',
            ecolor='black',
            capsize=14
        )

    ci_low, ci_high = 0.77, 0.85
    ax.axhspan(ci_low, ci_high, facecolor='pink', alpha=0.8, zorder=0)
    ax.axhline((ci_low + ci_high) / 2, linestyle='--', linewidth=3, alpha=0.8, color='black', zorder=1)

    summary.to_csv(fig_data_file_pth + f'lesions.csv')
    
    ax.set_ylabel("R$^2$", fontsize=18)
    ax.set_ylim(0, max(summary['ci_upper']) * 1.1)
    ax.set_xticks([])
    ax.tick_params(axis='x', bottom=False)
    ax.tick_params(axis='y', labelsize=14)

    fig.subplots_adjust(bottom=0.35)  
    ax_dots = fig.add_axes([0.125, 0.02, 0.775, 0.18])  

    max_space = 3.5
    dot_y_positions = np.linspace(3, 0, len(components))  
    for k, comp in enumerate(components):
        for j, config in enumerate(config_order):
            is_present = comp in component_map[config]
            color = 'black' if is_present else 'lightgray'
            ax_dots.scatter(j, dot_y_positions[k], color=color, s=80)

    ax_dots.set_xlim(-0.5, len(config_order) - 0.5)
    ax_dots.set_ylim(-0.5, max(dot_y_positions)+ 0.25)
    ax_dots.set_xticks(x)
    ax_dots.set_xticklabels(['' for _ in config_order])
    ax_dots.set_yticks(dot_y_positions)
    ax_dots.set_yticklabels(components, fontsize=12)
    ax_dots.tick_params(left=False, bottom=False)
    for spine in ax_dots.spines.values():
        spine.set_visible(False)
        
    plt.savefig('final_figures/param_fits/ablation_grid.pdf', dpi=300, bbox_inches='tight')
    plt.show()
